# Step 5 · ML model: predicting next season's teammate gap

**Inputs:** `data/teammate_gaps.csv`, `qualifying_clean.csv` · **Outputs:** `data/ml_features.csv`, `data/ml_predictions.csv`, two figures

**Task.** For a pair of teammates in season *s*, predict their mean qualifying gap (outliers removed, as in notebook 03) **using only information from seasons before s**.
One row = one usable pair-season (≥ 8 sessions), 2015–2026 (2014 is only used to build features).

**Fair testing (no leakage).**
- *Validation:* expanding window. For each season s = 2016 … 2024, train on all seasons before s and predict s. All choices (which model) are made here.
- *Test:* 2025 and 2026, predicted **once** at the end with the chosen method (each trained only on earlier seasons).

**Metrics (report Table 6):** MAE and RMSE (average error, in percentage points of lap time), direction accuracy (picks the faster teammate), and coverage of the 95% prediction interval.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance

from f1_stats import (load_gaps, pair_season_table, estimate_tau2, find_file, fit_ratings,
                      predicted_gap, setup_plot_style, TEAM_LINEAGE, REG_CHANGE_SEASONS,
                      BLUE, ORANGE, MUTED)

setup_plot_style()
os.makedirs('data', exist_ok=True)
os.makedirs('figures', exist_ok=True)

gaps = load_gaps()
ps = pair_season_table(gaps)
usable = ps[ps['usable']]
tau2, _ = estimate_tau2(ps)
quali = pd.read_csv(find_file('qualifying_clean.csv'))
print(f'{len(usable)} usable pair-seasons, tau^2 = {tau2:.4f}')

## 1. Features (all computed from earlier seasons only)

| Feature | Meaning | Why it might help |
|---|---|---|
| `prior_f` | Gap predicted by the weighted-least-squares ratings (notebook 04), fitted on seasons before s only | Links the pair through other drivers, works for new pairs |
| `prior_se_f` | Uncertainty of that prediction | Tells the model how much to trust `prior_f` |
| `last_f` | The pair's gap last season (if new pair: `prior_f`, else 0) | Gaps are fairly stable year to year |
| `has_last`, `has_prior` | Whether those exist | New pairs behave differently |
| `age_diff` | Age of A − age of B at the season's first race | Young drivers tend to improve |
| `exp_diff` | Seasons in our data (since 2014) of A − of B | Experience |
| `rookie_diff` | Rookie A (1/0) − rookie B (1/0) | First seasons are hardest |
| `newteam_diff` | Changed team this year: A (1/0) − B (1/0) | Adapting to a new car |
| `years_together` | Earlier seasons the two were teammates | Settled pairs change less |
| `reg_change` | 1 in 2017, 2022, 2026 | New rules shake up the order |

`*_diff` features flip sign when A and B are swapped, like the gap itself.

In [ ]:
quali['lineage'] = quali['constructorId'].map(TEAM_LINEAGE).fillna(quali['constructorId'])
seasons_of = quali.groupby('driverId')['year'].apply(lambda s: sorted(set(s)))
age_at_start = quali.groupby(['driverId', 'year'])['age'].min()
team_of = quali.groupby(['driverId', 'year'])['lineage'].agg(lambda s: s.mode().iat[0])
seasons_together = gaps.groupby(['A', 'B'])['year'].apply(lambda s: sorted(set(s)))

def experience(driver, s):
    return sum(1 for y in seasons_of[driver] if y < s)

def changed_team(driver, s):
    return int(team_of.get((driver, s - 1)) != team_of.get((driver, s)))

rows = []
for s in range(2015, 2027):
    for r in usable[usable['year'] == s].itertuples():
        # ratings fitted on seasons BEFORE s only, anchored on driver A
        prior, prior_se = predicted_gap(fit_ratings(ps, s, tau2, use_future=False, reference=r.A), r.A, r.B)
        last = usable[(usable['year'] == s - 1) & (usable['A'] == r.A) & (usable['B'] == r.B)]['mean']
        rows.append({
            'year': s, 'team': r.team, 'A': r.A, 'B': r.B, 'y': r.mean,
            'prior': prior, 'prior_se': prior_se,
            'last': last.iat[0] if len(last) else np.nan,
            'age_diff': age_at_start[(r.A, s)] - age_at_start[(r.B, s)],
            'exp_diff': experience(r.A, s) - experience(r.B, s),
            'rookie_diff': int(experience(r.A, s) == 0) - int(experience(r.B, s) == 0),
            'newteam_diff': changed_team(r.A, s) - changed_team(r.B, s),
            'years_together': sum(1 for y in seasons_together.get((r.A, r.B), []) if y < s),
            'reg_change': int(s in REG_CHANGE_SEASONS),
        })

F = pd.DataFrame(rows)
F['has_prior'] = F['prior'].notna().astype(int)
F['has_last'] = F['last'].notna().astype(int)
F['prior_f'] = F['prior'].fillna(0)
F['prior_se_f'] = F['prior_se'].fillna(1.0)
F['last_f'] = F['last'].fillna(F['prior']).fillna(0)

print(f'{len(F)} rows | continuing pairs: {F.has_last.sum()} | new pairs: {(1 - F.has_last).sum()} '
      f'| with a prior estimate: {F.has_prior.sum()}')
F.to_csv('data/ml_features.csv', index=False)
F.head()

## 2. Candidates

**Baselines** (from the report): guess 0 ("teammates are equal") and last season's gap (0 for a new pair). Also the **statistical prior** on its own.

**ML models:** Ridge regression (features standardised inside a pipeline, so the scaler only sees training seasons), Random Forest and Gradient Boosting.

**Hybrid:** last season's gap when the pair existed last season, otherwise the Random Forest.

**Mirror trick:** swapping A and B must flip the prediction. Each training row is also added mirrored (gap and `*_diff` features × −1), which doubles the small training set and makes the models respect this symmetry.

In [ ]:
FLIP = ['prior_f', 'last_f', 'age_diff', 'exp_diff', 'rookie_diff', 'newteam_diff']
KEEP = ['has_prior', 'has_last', 'prior_se_f', 'years_together', 'reg_change']
FEATURES = FLIP + KEEP

def mirrored(d):
    m = d.copy()
    m[FLIP] = -m[FLIP]
    m['y'] = -m['y']
    return pd.concat([d, m], ignore_index=True)

MODELS = {
    'Ridge': lambda: make_pipeline(StandardScaler(), Ridge(alpha=30)),
    'Random forest': lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0),
    'Gradient boosting': lambda: GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                                           max_depth=2, subsample=0.8, random_state=0),
}

def walk_forward(seasons):
    """For each season s: train every model on seasons < s, predict s. Returns all predictions."""
    out = []
    for s in seasons:
        train, test = mirrored(F[F['year'] < s]), F[F['year'] == s].copy()
        test['Guess 0'] = 0.0
        test["Last season's gap"] = test['last'].fillna(0)
        test['Statistical prior'] = test['prior'].fillna(0)
        for name, make in MODELS.items():
            model = make().fit(train[FEATURES], train['y'])
            test[name] = model.predict(test[FEATURES])
        test['Hybrid'] = np.where(test['has_last'] == 1, test['last'], test['Random forest'])
        out.append(test)
    return pd.concat(out, ignore_index=True)

CANDIDATES = ['Guess 0', "Last season's gap", 'Statistical prior', 'Ridge', 'Random forest',
              'Gradient boosting', 'Hybrid']

def scores(d):
    rows = []
    for c in CANDIDATES:
        err = d['y'] - d[c]
        picks = d[c] != 0
        rows.append({'method': c, 'MAE': err.abs().mean(), 'RMSE': np.sqrt((err**2).mean()),
                     'direction': (np.sign(d.loc[picks, c]) == np.sign(d.loc[picks, 'y'])).sum() / len(d),
                     'MAE continuing pairs': err[d['has_last'] == 1].abs().mean(),
                     'MAE new pairs': err[d['has_last'] == 0].abs().mean()})
    return pd.DataFrame(rows).set_index('method')

## 3. Validation (2016–2024): choose the method here
Direction = share of pairs where the method names the faster teammate (guessing 0 never names one).

In [ ]:
val = walk_forward(range(2016, 2025))
val_scores = scores(val)
print(f'{len(val)} validation pair-seasons')
val_scores.round(3)

In [ ]:
best = val_scores['MAE'].idxmin()
residuals = val['y'] - val[best]
half_width = np.quantile(residuals.abs(), 0.95)     # 95% prediction interval = prediction ± this
print(f'Chosen method (lowest validation MAE): {best}')
print(f'95% prediction interval: prediction ± {half_width:.2f} percentage points')

## 4. Test (2025–2026), run once with the chosen method
The other methods are shown for reference only; the choice above was fixed before looking at these seasons.

In [ ]:
test = walk_forward([2025, 2026])
test['prediction'] = test[best]
test['inside_95'] = (test['y'] - test['prediction']).abs() <= half_width
err = test['y'] - test['prediction']
print(f"Test pair-seasons: {len(test)}")
print(f"{best}: MAE {err.abs().mean():.3f} | RMSE {np.sqrt((err**2).mean()):.3f} | "
      f"direction {(np.sign(test['prediction']) == np.sign(test['y'])).mean():.0%} | "
      f"coverage of 95% interval {test['inside_95'].mean():.0%}")
scores(test).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 5.8))
lim = 1.2
ax.plot([-lim, lim], [-lim, lim], color=MUTED, linewidth=1, label='perfect prediction')
ax.plot(val[best], val['y'], 'o', color=BLUE, markersize=6, alpha=0.75,
        markeredgecolor='white', markeredgewidth=0.8, label='validation 2016–2024')
ax.plot(test['prediction'], test['y'], 'o', color=ORANGE, markersize=7,
        markeredgecolor='white', markeredgewidth=0.8, label='test 2025–2026')
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel(f'Predicted gap, % ({best})')
ax.set_ylabel('Actual gap, %')
ax.set_title('Predicted vs actual teammate gap')
ax.legend(loc='upper left')
fig.tight_layout()
fig.savefig('figures/ml_pred_vs_actual.png', bbox_inches='tight')
plt.show()

## 5. Which features matter? (Random Forest, permutation importance)
For each validation season: fit on earlier seasons, then shuffle one feature at a time and measure how much the MAE gets worse. Averaged over the nine validation seasons.

In [ ]:
imp = []
for s in range(2016, 2025):
    train, hold = mirrored(F[F['year'] < s]), F[F['year'] == s]
    model = MODELS['Random forest']().fit(train[FEATURES], train['y'])
    res = permutation_importance(model, hold[FEATURES], hold['y'], n_repeats=10, random_state=0,
                                 scoring='neg_mean_absolute_error')
    imp.append(res.importances_mean)
importance = pd.Series(np.mean(imp, axis=0), index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.barh(importance.index, importance.values, color=BLUE, height=0.6)
ax.axvline(0, color=MUTED, linewidth=1)
ax.grid(axis='y', visible=False)
ax.set_xlabel('Increase in MAE when the feature is shuffled')
ax.set_title('Feature importance (Random Forest)')
fig.tight_layout()
fig.savefig('figures/ml_feature_importance.png', bbox_inches='tight')
plt.show()
importance.sort_values(ascending=False).round(4)

## What this shows (our run)

- **Validation 2016–2024 (95 pair-seasons):** the Hybrid had the lowest MAE, 0.22, against 0.23 for last season's gap and 0.28 for guessing 0. It named the faster teammate 72% of the time.
- **Test 2025–2026 (21 pair-seasons, run once):** Hybrid MAE 0.16 vs 0.19 (last season) and 0.23 (guess 0), RMSE 0.21, faster teammate right 86% of the time, and all 21 real gaps inside the ±0.59 prediction interval (the interval is wide and probably conservative with so few test pairs).
- **Where ML helps:** for pairs that existed last season, simply copying last season's gap is already hard to beat. The Random Forest earns its place on **new pairs**, where there is no last season (validation MAE 0.29 vs 0.31 for guessing 0).
- **What drives the predictions:** the statistical prior and last season's gap dominate; age difference, rookie status and experience add a little; team changes, regulation changes and years together add almost nothing.
- **Caveat:** about 10 pairs per season, so the test set is small and these numbers will move with a few pairs.

## 6. Save

In [ ]:
keep = ['year', 'team', 'A', 'B', 'y', 'has_last'] + CANDIDATES
pd.concat([val[keep].assign(split='validation'),
           test[keep + ['inside_95']].assign(split='test')]).to_csv('data/ml_predictions.csv', index=False)
print('saved data/ml_predictions.csv')